# Phase 9.5 — Fix WaveGenerator Training

Phase 9 trained three generation modules. Stage 1 (WaveToText) succeeded.
Stage 2 (WaveGenerator) mode-collapsed. Stage 3 (Joint FT) silently skipped 100% of steps.

Phase 9.5 reloads working components from `phase9.phase.pt` and retrains
WaveGenerator from scratch with six fixes:

1. **Context Collapse** → L2-normalize + noise augmentation + 5× contrastive loss
2. **Fixed 40 Chunks** → Random windows + variable lengths
3. **Batch Size 1** → DataLoader batch_size=128 + batched GRU forward
4. **SS at 0%** → Start at 50% from step 1, ramp to 90%
5. **Train-Inference Mismatch** → Gaussian jitter on precomputed contexts
6. **Silent Error Swallowing** → Log tracebacks, abort on >10% skip rate

**Acceptance Criteria:**
- Wave 0 cross-context cosine < 0.85 (was 1.000)
- Hidden init cross-context cosine < 0.90 (was 0.996)
- Free generation produces recognizable English words
- GPU utilization > 60% (was 17%)
- Training speed > 200 steps/s (was 17.4)
- No silently skipped steps

## Cell 1 — Clone / Pull Repo

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
REPO_DIR = '/kaggle/working/FLUX'

if os.path.exists(REPO_DIR) and os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('  ℹ Repo exists, pulling latest...')
    r = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR,
                       capture_output=True, text=True, timeout=60)
    print(r.stdout[:200] if r.returncode == 0 else f'  ⚠ Pull failed: {r.stderr[:100]}')
else:
    print(f'  ℹ Cloning {REPO_URL}...')
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                       capture_output=True, text=True, timeout=120)
    if r.returncode == 0:
        print(f'  ✓ Cloned to {REPO_DIR}')
    else:
        raise RuntimeError(f'Clone failed: {r.stderr}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'  ✓ Working dir: {os.getcwd()}')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt

import torch
print(f'  ✓ PyTorch {torch.__version__}')
print(f'  ✓ CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  ✓ GPU: {torch.cuda.get_device_name(0)}')
    print(f'  ✓ VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Cell 3 — Init Logger + Hardware + Secrets

In [ ]:
import sys
from pathlib import Path

# Path setup for all phases
PHASES_DIR = Path('phases')
for phase_name in ['phase1', 'phase2', 'phase3', 'phase4',
                    'phase5', 'phase6', 'phase7', 'phase8',
                    'phase9', 'phase9_5']:
    p = str(PHASES_DIR / phase_name)
    if p not in sys.path:
        sys.path.insert(0, p)
sys.path.insert(0, '.')

from flux_utils import PhaseLogger, get_device, _resolve_hf_token

log = PhaseLogger(phase=9.5)
log.cell_start('Cell 3 — Hardware & Secrets')

device = get_device()
log.info(f'Device: {device}')

# Resolve HF token
hf_token = _resolve_hf_token()
if hf_token:
    log.success(f'HF token resolved ({len(hf_token)} chars)')
else:
    log.warning('No HF token — checkpoint upload will be skipped')

if device == 'cuda':
    import torch
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    log.info(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
    if gpu_mem < 8:
        log.warning('< 8GB VRAM — may need smaller batch size')

log.cell_end('Cell 3 — Hardware & Secrets', 'PASS')

## Cell 4 — Smoke Test: Verify Phase 9 Checkpoint

In [ ]:
log.cell_start('Cell 4 — Smoke Test')

from flux_utils import checkpoint_exists, load_checkpoint

assert checkpoint_exists(9), (
    'Phase 9 checkpoint not found! '
    'Download from HuggingFace or run Phase 9 first.'
)

ckpt = load_checkpoint(9)
log.success(f'Phase 9 checkpoint loaded')
log.info(f'Keys: {list(ckpt.keys())[:10]}...')
log.info(f'Phase: {ckpt.get("phase")}')

# Verify key components exist
required_keys = [
    'cse_state_dict', 'field_state_dict', 'wave_chunker_state_dict',
    'wave_to_text_state_dict', 'wave_generator_state_dict',
]
for key in required_keys:
    assert key in ckpt, f'Missing key: {key}'
    log.success(f'  {key}: present')

del ckpt  # Free memory
log.cell_end('Cell 4 — Smoke Test', 'PASS')

## Cell 5 — Load Phase 9 Components

Load all frozen components from `phase9.phase.pt`.
WaveGenerator state is **discarded** — we retrain from scratch.

In [ ]:
log.cell_start('Cell 5 — Load Phase 9')

from train_wave_gen_v2 import (
    load_from_phase9_checkpoint,
    build_fresh_wave_generator,
    PHASE9_5_CONFIG,
)

model, chunker, wtt, ckpt9 = load_from_phase9_checkpoint(device=device)
log.success('Phase 9 components loaded (WaveGenerator DISCARDED)')

# Build fresh WaveGeneratorV3
generator = build_fresh_wave_generator(device=device)
log.success('Fresh WaveGeneratorV3 built')

log.cell_end('Cell 5 — Load Phase 9', 'PASS')

## Cell 6 — Load Training Data

In [ ]:
log.cell_start('Cell 6 — Training Data')

from train_wave_gen_v2 import load_training_data

texts = load_training_data(max_docs=10000)
split_idx = int(len(texts) * 0.9)
train_texts = texts[:split_idx]
eval_texts = texts[split_idx:]
log.info(f'Train: {len(train_texts):,} docs, Eval: {len(eval_texts):,} docs')

log.cell_end('Cell 6 — Training Data', 'PASS')

## Cell 7a — Precompute (FIX #2: Random Windows + Variable Lengths)

In [ ]:
log.cell_start('Cell 7a — Precompute')

from train_wave_gen_v2 import precompute_wg_data

precomputed = precompute_wg_data(
    model, chunker, train_texts,
    max_samples=8500, device=device,
)

# Verify variable lengths
lengths = [p[1].shape[0] for p in precomputed]
print(f'\n  Precomputed: {len(precomputed):,} samples')
print(f'  Lengths: min={min(lengths)}, max={max(lengths)}, unique={len(set(lengths))}')
print(f'  Sample shape: merged={precomputed[0][0].shape}, target_waves={precomputed[0][1].shape}')

log.cell_end('Cell 7a — Precompute', 'PASS')

## Cell 7b — Precompute Sanity Check

In [ ]:
log.cell_start('Cell 7b — Precompute Sanity')
import torch.nn.functional as F

# Check context diversity before training
n_check = min(20, len(precomputed))
ctxs = torch.stack([precomputed[i][0] for i in range(n_check)])
cos_matrix = F.cosine_similarity(ctxs.unsqueeze(0), ctxs.unsqueeze(1), dim=-1)
mask = ~torch.eye(n_check, dtype=torch.bool)
avg_cos = cos_matrix[mask].mean().item()
print(f'  Raw context avg pairwise cosine: {avg_cos:.3f} (Phase 9 was 0.980)')

# NaN/Inf check
nan_count = sum(1 for m, t in precomputed[:100] if torch.isnan(m).any() or torch.isnan(t).any())
inf_count = sum(1 for m, t in precomputed[:100] if torch.isinf(m).any() or torch.isinf(t).any())
print(f'  NaN tensors: {nan_count}  |  Inf tensors: {inf_count}')
assert nan_count == 0 and inf_count == 0, 'Found NaN/Inf in precomputed data!'
print(f'  ✓ All clean')

log.cell_end('Cell 7b — Precompute Sanity', 'PASS')

## Cell 8a — Stage 1: WaveGenerator Training

Applies all six fixes:
- FIX #1: L2-normalize + noise augmentation + 5× contrastive loss
- FIX #2: Random windows (already applied in precompute)
- FIX #3: Batched DataLoader, batch_size=128
- FIX #4: Scheduled sampling 50%→90% from step 1
- FIX #5: Gaussian jitter on contexts (in DataLoader)

In [ ]:
log.cell_start('Cell 8a — WG Train')

from train_wave_gen_v2 import train_wave_generator

wg_result = train_wave_generator(
    generator, precomputed,
    max_steps=15000,
    batch_size=128,
    lr=3e-4,
    ss_start=0.5,
    ss_end=0.9,
    contrastive_weight=5.0,
    noise_sigma=0.1,
    log_interval=500,
    device=device,
    log=log,
)

log.metric('wg_final_loss', f'{wg_result.final_loss:.4f}')
log.metric('wg_cos_acc', f'{wg_result.wave_cosine_accuracy:.3f}')
log.metric('wg_speed', f'{wg_result.steps_per_second:.1f} step/s')
log.cell_end('Cell 8a — WG Train', 'PASS')

## Cell 8b — WG Diagnostic: Context Diversity

In [ ]:
log.cell_start('Cell 8b — WG Diag')

from train_wave_gen_v2 import evaluate_context_diversity

diversity = evaluate_context_diversity(
    generator, precomputed, n_samples=20, device=device
)

w0_cos = diversity['wave0_cross_ctx_cosine']
h_cos = diversity['hidden_init_cross_ctx_cosine']

print(f'\n  Acceptance:')
print(f'    Wave 0 cross-ctx cosine: {w0_cos:.3f} {"✓" if w0_cos < 0.85 else "✗"} (target <0.85)')
print(f'    Hidden init cross-ctx:   {h_cos:.3f} {"✓" if h_cos < 0.90 else "✗"} (target <0.90)')

log.metric('wave0_cross_ctx_cosine', f'{w0_cos:.3f}')
log.metric('hidden_init_cross_ctx_cosine', f'{h_cos:.3f}')
log.cell_end('Cell 8b — WG Diag', 'PASS' if w0_cos < 0.85 else 'WARN')

## Cell 9a — Stage 2: Joint Fine-Tuning (FIX #6: No Silent Errors)

In [ ]:
log.cell_start('Cell 9a — Joint FT')

from train_wave_gen_v2 import train_joint_finetune

joint_result = train_joint_finetune(
    generator, wtt, model, chunker, train_texts, precomputed,
    max_steps=3000,
    lr_wg=1e-4,
    lr_wtt=5e-5,
    log_interval=500,
    max_skip_rate=0.10,
    device=device,
    log=log,
)

log.metric('joint_steps', joint_result.total_steps)
log.metric('joint_skipped', joint_result.skipped_steps)
log.metric('joint_cos_acc', f'{joint_result.wave_cosine_accuracy:.3f}')
log.metric('joint_wtt_acc', f'{joint_result.wtt_word_accuracy:.1%}')

# FIX #6: Verify no silent skipping
if joint_result.total_steps == 0:
    log.error('Joint FT did ZERO work — investigate errors above')
    log.cell_end('Cell 9a — Joint FT', 'FAIL')
else:
    skip_rate = joint_result.skipped_steps / (joint_result.total_steps + joint_result.skipped_steps)
    log.info(f'Skip rate: {skip_rate:.1%}')
    log.cell_end('Cell 9a — Joint FT', 'PASS')

## Cell 9b — Free Generation Evaluation

In [ ]:
log.cell_start('Cell 9b — Free Gen Eval')

from train_wave_gen_v2 import generate_text

prompts = [
    'The future of artificial intelligence',
    'Scientists have discovered',
    'In the beginning',
    'The relationship between energy and matter',
    'Modern technology relies on',
]

valid_words = 0
total_words = 0

for p in prompts:
    try:
        result = generate_text(p, model, chunker, generator, wtt, max_waves=15)
        continuation = result[len(p):].strip()
        words = continuation.split()
        for w in words:
            clean = w.strip('.,;:!\'\"?-()[]{}').lower()
            if clean.isalpha() and len(clean) >= 2:
                total_words += 1
                if len(clean) <= 15:
                    valid_words += 1
        print(f'  Prompt: {p}')
        print(f'  Output: "{result[:200]}"')
        print()
    except Exception as e:
        print(f'  ⚠ Generation failed for "{p[:30]}...": {e}')

valid_rate = valid_words / max(total_words, 1)
print(f'  Valid word rate: {valid_words}/{total_words} ({valid_rate:.1%})')
log.metric('valid_word_rate', f'{valid_rate:.1%}')
log.cell_end('Cell 9b — Free Gen Eval', 'PASS' if valid_rate > 0.3 else 'WARN')

## Cell 10 — Save Checkpoint

In [ ]:
log.cell_start('Cell 10 — Save Checkpoint')

from train_wave_gen_v2 import save_phase9_5_checkpoint

ckpt_path = save_phase9_5_checkpoint(
    model, chunker, generator, wtt,
    wg_result, joint_result, diversity, valid_rate,
)

log.success(f'Checkpoint saved: {ckpt_path}')
log.cell_end('Cell 10 — Save Checkpoint', 'PASS')

## Cell 11 — Upload to HuggingFace Hub

In [ ]:
log.cell_start('Cell 11 — HF Upload')

from flux_utils import upload_checkpoint_to_hf

if hf_token:
    # Upload as phase9_5
    from flux_utils import CHECKPOINT_DIR, HF_REPO_ID
    from huggingface_hub import HfApi
    try:
        api = HfApi(token=hf_token)
        api.upload_file(
            path_or_fileobj=str(CHECKPOINT_DIR / 'phase9_5.phase.pt'),
            path_in_repo='checkpoints/phase9_5.phase.pt',
            repo_id=HF_REPO_ID,
            commit_message=f'Phase 9.5 checkpoint — WaveGenerator fix',
        )
        log.success('Checkpoint uploaded to HuggingFace Hub')
    except Exception as e:
        log.warning(f'HF upload failed: {e}')
else:
    log.warning('No HF token — skipping upload')

log.cell_end('Cell 11 — HF Upload', 'PASS')

## Cell 12 — Test 1: Context Collapse

In [ ]:
log.cell_start('Cell 12 — Test 1')

w0_cos = diversity['wave0_cross_ctx_cosine']
h_cos = diversity['hidden_init_cross_ctx_cosine']

t1_pass = w0_cos < 0.85 and h_cos < 0.90

print(f'  Wave 0 cross-ctx cosine: {w0_cos:.3f} (target <0.85)  {"✓" if w0_cos < 0.85 else "✗"}')
print(f'  Hidden init cross-ctx:   {h_cos:.3f} (target <0.90)  {"✓" if h_cos < 0.90 else "✗"}')
print(f'  Test 1: {"PASS" if t1_pass else "FAIL"}')

log.cell_end('Cell 12 — Test 1', 'PASS' if t1_pass else 'FAIL')

## Cell 13 — Test 2: Training Speed

In [ ]:
log.cell_start('Cell 13 — Test 2')

speed = wg_result.steps_per_second
t2_pass = speed > 200

print(f'  Training speed: {speed:.1f} steps/s (target >200, Phase 9 was 17.4)')
print(f'  Test 2: {"PASS" if t2_pass else "FAIL"}')

log.cell_end('Cell 13 — Test 2', 'PASS' if t2_pass else 'FAIL')

## Cell 14 — Test 3: Generation Quality

In [ ]:
log.cell_start('Cell 14 — Test 3')

t3_pass = valid_rate > 0.3

print(f'  Valid word rate: {valid_rate:.1%} (target >30%)')
print(f'  Phase 9 output: "a s y  y   u u m t Fk  u  u n h  u  u at t  t"')
print(f'  Test 3: {"PASS" if t3_pass else "FAIL"}')

log.cell_end('Cell 14 — Test 3', 'PASS' if t3_pass else 'FAIL')

## Cell 15 — Results Summary

In [ ]:
log.cell_start('Cell 15 — Results')

from flux_utils import PhaseResults

results = PhaseResults(phase=9.5, component_name='WaveGenerator Fix')

results.add_test('Wave 0 cross-ctx cosine', w0_cos < 0.85,
                 score=f'{w0_cos:.3f}', threshold='< 0.85')
results.add_test('Hidden init cross-ctx cosine', h_cos < 0.90,
                 score=f'{h_cos:.3f}', threshold='< 0.90')
results.add_test('Training speed', wg_result.steps_per_second > 200,
                 score=f'{wg_result.steps_per_second:.1f}', threshold='> 200 step/s')
results.add_test('Valid word rate', valid_rate > 0.3,
                 score=f'{valid_rate:.1%}', threshold='> 30%')
results.add_test('No silent skips', joint_result.total_steps > 0,
                 score=f'{joint_result.total_steps} steps', threshold='> 0')

results.add_metric('WG steps', wg_result.total_steps)
results.add_metric('WG final loss', f'{wg_result.final_loss:.4f}')
results.add_metric('WG cosine acc', f'{wg_result.wave_cosine_accuracy:.3f}')
results.add_metric('WG speed', f'{wg_result.steps_per_second:.1f} step/s')
results.add_metric('Joint steps', joint_result.total_steps)
results.add_metric('Joint skipped', joint_result.skipped_steps)
results.add_metric('Valid word rate', f'{valid_rate:.1%}')

results.save('phases/phase9_5/RESULTS_PHASE_9_5.md')

log.cell_end('Cell 15 — Results', 'PASS' if results.all_tests_passed() else 'FAIL')

## Cell 16 — View Full Log

In [ ]:
print(log.get_log_contents())

## Cell 17 — Final Upload (Logs → HF + GitHub)

In [ ]:
log.cell_start('Cell 17 — Final Upload')

from flux_utils import upload_logs_to_hf, git_commit_and_push

if hf_token:
    upload_logs_to_hf(phase=9.5, hf_token=hf_token)

git_commit_and_push(
    files=['logs/phase9.5.log', 'phases/phase9_5/RESULTS_PHASE_9_5.md'],
    message='Phase 9.5 — WaveGenerator fix results',
)

log.cell_end('Cell 17 — Final Upload', 'PASS')
log.success('Phase 9.5 complete!')

## Cell 18 — Save Artifacts to Kaggle Output

In [ ]:
import shutil
from pathlib import Path

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# Copy checkpoint
ckpt_src = Path('checkpoints/phase9_5.phase.pt')
if ckpt_src.exists():
    shutil.copy2(ckpt_src, output_dir / 'phase9_5.phase.pt')
    print(f'  ✓ Checkpoint copied to output/')

# Copy results
results_src = Path('phases/phase9_5/RESULTS_PHASE_9_5.md')
if results_src.exists():
    shutil.copy2(results_src, output_dir / 'RESULTS_PHASE_9_5.md')
    print(f'  ✓ Results copied to output/')

# Copy log
log_src = Path('logs/phase9.5.log')
if log_src.exists():
    shutil.copy2(log_src, output_dir / 'phase9_5.log')
    print(f'  ✓ Log copied to output/')

print(f'\n  Output artifacts: {list(output_dir.iterdir())}')